> 在线讲义：[第 17 章 · 函数近似与梯度下降：表格装不下，换成一个会"猜"的函数](https://wikieden.github.io/rl-from-scratch/lessons/17-function-approx.html) · 本书《从零学强化学习》

# 第 17 章 · 函数近似与梯度下降：表格装不下，换成一个会"猜"的函数

本章回答：状态多到表格装不下时（围棋、机器人的连续关节角），价值怎么存？"梯度下降"是什么？怎么用它把一个函数调得越来越准？

配套讲义：仓库里 `cs285-deep-rl/notes/lessons/17-function-approx.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import numpy as np
import torch

np.set_printoptions(precision=3, suppress=True)  # 打印保留 3 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 代码 | 跑出来 |
|---|---|---|
| 斜率 | `slope(lambda x: x * x, 3.0)` | 6.0 |
| 梯度下降 | `gradient_descent()` | 0 → 0.6 → 1.08 → 1.464 → … → 2.996 |
| PyTorch 自动求导 | `(x**2).backward(); x.grad` | 6.0 |
| 线性梯度 MC | `gradient_mc_linear()` | w = [0.5, 0.341]，五个估计 0.159 … 0.842 |

## 1. 先把要用的东西定义好

这几格只是定义（环境、函数），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `WALK_TRUE`**（来自 `rl_envs.py`）

In [ ]:
WALK_TRUE = np.array([1, 2, 3, 4, 5]) / 6

**函数 `features`**（来自 `ch17_function_approx.py`）：随机游走 5 个状态，用 2 个特征 [1, 位置] 表示（位置 A=−1 … E=+1）。

In [ ]:
def features(s):
    """随机游走 5 个状态，用 2 个特征 [1, 位置] 表示（位置 A=−1 … E=+1）。"""
    return np.array([1.0, (s - 2) / 2])

**函数 `gradient_descent`**（来自 `ch17_function_approx.py`）：最小化 (w − 3)²：导数 2(w − 3)，每步 w ← w − lr · 导数。

In [ ]:
def gradient_descent(w0=0.0, lr=0.1, steps=30):
    """最小化 (w − 3)²：导数 2(w − 3)，每步 w ← w − lr · 导数。"""
    w, hist = w0, [w0]
    for _ in range(steps):
        w -= lr * 2 * (w - 3)
        hist.append(w)
    return hist

**函数 `random_walk_episode`**（来自 `rl_envs.py`）：从 C 出发，每步等概率往左或往右；走出左边奖励 0，走出右边奖励 +1。

In [ ]:
def random_walk_episode(rng):
    """从 C 出发，每步等概率往左或往右；走出左边奖励 0，走出右边奖励 +1。
    返回 [(状态, 奖励, 下一个状态或 None)]，状态 0–4 对应 A–E。"""
    s, steps = 2, []
    while True:
        ns = s + (1 if rng.random() < 0.5 else -1)
        if ns < 0:
            steps.append((s, 0.0, None))
            return steps
        if ns > 4:
            steps.append((s, 1.0, None))
            return steps
        steps.append((s, 0.0, ns))
        s = ns

**函数 `gradient_mc_linear`**（来自 `ch17_function_approx.py`）：线性 v̂(s, w) = w · x(s)，梯度就是 x(s)。目标 U 用整局回报 G（梯度 MC，§9.3）。

In [ ]:
def gradient_mc_linear(n_episodes=5000, alpha=0.02, seed=0):
    """线性 v̂(s, w) = w · x(s)，梯度就是 x(s)。目标 U 用整局回报 G（梯度 MC，§9.3）。
    步长随局数慢慢变小（α_k = α / (1 + k/100)），否则 w 会一直在真值附近抖。"""
    rng, w = np.random.default_rng(seed), np.zeros(2)
    for k in range(n_episodes):
        a = alpha / (1 + k / 100)
        ep = random_walk_episode(rng)
        G = ep[-1][1]  # γ = 1，中间奖励都是 0，回报就是最后那个奖励
        for s, _, _ in ep:
            x = features(s)
            w += a * (G - w @ x) * x
    return w

**函数 `slope`**（来自 `ch17_function_approx.py`）：数值求导：往右挪一点点，看 f 变了多少。

In [ ]:
def slope(f, x, h=1e-4):
    """数值求导：往右挪一点点，看 f 变了多少。"""
    return (f(x + h) - f(x - h)) / (2 * h)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("f(x) = x² 在 x = 3 的斜率 ≈", round(slope(lambda x: x * x, 3.0), 4))

In [ ]:
hist = gradient_descent()

In [ ]:
print("梯度下降 (w − 3)²：", [round(h, 3) for h in hist[:5]], "…", round(hist[-1], 4))

In [ ]:
x = torch.tensor(3.0, requires_grad=True)

In [ ]:
(x**2).backward()

In [ ]:
print("PyTorch autograd：d(x²)/dx 在 3 =", x.grad.item())

In [ ]:
w = gradient_mc_linear()

In [ ]:
print("线性近似 w =", np.round(w, 3), "；v̂ =", np.round([w @ features(s) for s in range(5)], 3))

In [ ]:
print("真值       ", np.round(WALK_TRUE, 3))

## 3. 代码里才看得到的

神经网络有几百万个参数，没人手算导数；PyTorch 的 `backward()` 会自动把每个参数的导数算好，放在 `.grad` 里。第 18 章训练 Q 网络、第 20 章训练策略网络，都是"算一个 loss → `backward()` → 优化器走一步"这三行。

## 4. 自己改一改

讲义里建议的几个小实验（数字是实跑出来的，改完对一对）：

1. `gradient_descent(lr=0.5, steps=6)` → 0、3、3、3…：一步就到底。`lr=0.9` → 0、5.4、1.08、4.536…：左右来回跳，慢慢收敛。`lr=1.1` → 0、6.6、−1.32、8.184…：越跳越远，发散了。
2. 把 `gradient_mc_linear(alpha=...)` 从 0.02 调到 0.2，看 w 还稳不稳。

讲义末尾的练习题也可以在下面的空格子里用代码验算：先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 状态太多不存表，带参数的函数来猜：$\hat v(s, \mathbf w)$；
> 导数是斜率，往右挪一点看变多少；多个参数排一排就是梯度；
> 梯度下降：朝下坡走一小步，坡陡步大、坡缓步小；
> 式 9.7：$\mathbf w$ += α × (目标 − 估计) × 梯度；TD 目标不求导叫半梯度。